In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict

In [10]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
IPS = DATA_INITIAL / "deg/rnaseq_res_ips"
FIBRO = DATA_INITIAL / "deg/rnaseq_res_fibro"
IPS_INDEX = PREPROCESSING / "deg_index/ips_deg_index"
FIBRO_INDEX = PREPROCESSING / "deg_index/fibro_deg_index"
BIOMART = DATA / "biomart"
BIOMART_INITIAL = DATA_INITIAL / "biomart"
GENCODE_INITIAL = DATA_INITIAL / "gencode"

In [4]:
to_ips = IPS / "EdgeR.table.all.target.tsv.gz"
ips_expr = pd.read_csv(to_ips, sep="\t")
ips_expr.head()

,DEG.geneID,DEG.HGNC,EdgeR.log2FC,EdgeR.log2CPM,EdgeR.pvalue,EdgeR.FDR,EdgeR.Zscore,target.gapmerID,target.geneID,target.HGNC
0,CATG00000000155,CATG00000000155.1,0.257141,3.591144,0.550588,1.0,0.620927,G0234787_01,ENSG00000234787,LINC00458
1,CATG00000001164,CATG00000001164.1,0.706131,3.805757,0.044323,1.0,1.757448,G0234787_01,ENSG00000234787,LINC00458
2,CATG00000001318,CATG00000001318.1,-0.074655,4.766566,0.913694,1.0,0.052701,G0234787_01,ENSG00000234787,LINC00458
3,CATG00000002817,CATG00000002817.1,-0.201767,3.841792,0.716487,1.0,-0.414103,G0234787_01,ENSG00000234787,LINC00458
4,CATG00000003248,CATG00000003248.1,-0.155609,4.592817,0.650460,1.0,0.026003,G0234787_01,ENSG00000234787,LINC00458


In [5]:
ips_expr["DEG.geneID"].nunique()

10868

In [6]:
to_fibro = FIBRO / "DESeq2_genes_ASO_all.tsv.gz"
fibro_expr = pd.read_csv(to_fibro, sep="\t")
fibro_expr.head(3)

,Description,ASO_CAGE,KD.geneSymbol,perturb_id,geneID,geneSymbol,baseMean,log2FC,pvalue,fdr,KD.tpm.ave,NC.tpm.ave,ZScore
0,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000200169,RNU5D-1,134.656674,-2.325717,1.640034e-11,2.548449e-07,3.026571,15.427402,-3.106205
1,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000138675,FGF5,1788.727654,-1.371879,7.814666e-11,6.071605e-07,75.054256,196.439355,-2.756217
2,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000252521,RNU5D-2P,108.388065,-1.925674,1.102746e-09,5.711854e-06,3.178508,12.249495,-2.534245


In [33]:
fibro_expr["geneID"].nunique()

21041

In [11]:
gtf_file_GENCODE = GENCODE_INITIAL / "gencode.v47.basic.annotation.gtf.gz" #
anno_GENCODE = pd.read_csv(gtf_file_GENCODE, sep="\t", header=None, skiprows=5)
anno_GENCODE.head(2)

,0,1,2,3,4,5,6,7,8
0,chr1,HAVANA,gene,11121,24894,.,+,.,"gene_id ""ENSG00000290825.2""; gene_type ""lncRNA..."
1,chr1,HAVANA,transcript,11426,14409,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."


In [12]:
anno_GENCODE

,0,1,2,3,4,5,6,7,8
0,chr1,HAVANA,gene,11121,24894,.,+,.,"gene_id ""ENSG00000290825.2""; gene_type ""lncRNA..."
1,chr1,HAVANA,transcript,11426,14409,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
2,chr1,HAVANA,exon,11426,11671,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
3,chr1,HAVANA,exon,12010,12227,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
4,chr1,HAVANA,exon,12613,12721,.,+,.,"gene_id ""ENSG00000290825.2""; transcript_id ""EN..."
...,...,...,...,...,...,...,...,...,...
2225573,chrM,ENSEMBL,transcript,15888,15953,.,+,.,"gene_id ""ENSG00000210195.2""; transcript_id ""EN..."
2225574,chrM,ENSEMBL,exon,15888,15953,.,+,.,"gene_id ""ENSG00000210195.2""; transcript_id ""EN..."
2225575,chrM,ENSEMBL,gene,15956,16023,.,-,.,"gene_id ""ENSG00000210196.2""; gene_type ""Mt_tRN..."
2225576,chrM,ENSEMBL,transcript,15956,16023,.,-,.,"gene_id ""ENSG00000210196.2""; transcript_id ""EN..."


In [13]:
anno_GENCODE["ens_gene"] = anno_GENCODE[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])

In [14]:
anno_GENCODE["unique_genome_reg_name"] = anno_GENCODE["ens_gene"] + "-" + anno_GENCODE[2]

In [22]:
gencode_hg38 = anno_GENCODE[[0, 3, 4, "unique_genome_reg_name", 5, 6, "ens_gene"]]

In [23]:
mask = gencode_hg38["unique_genome_reg_name"].apply(lambda x: "gene" in x)
gencode_genes_hg38 = gencode_hg38[mask]
gencode_genes_hg38.head(2)

,0,3,4,unique_genome_reg_name,5,6,ens_gene
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825
6,chr1,12010,13670,ENSG00000223972-gene,.,+,ENSG00000223972


In [24]:
gencode_genes_hg38 = gencode_genes_hg38.rename(columns={0:"chrom", 3:"start", 4:"end", "unique_genome_reg_name":"name", 5:"score", 6:"strand"})

In [25]:
gencode_genes_hg38.shape

(78724, 7)

In [26]:
gencode_genes_hg38 = gencode_genes_hg38.drop_duplicates()
gencode_genes_hg38.shape

(78724, 7)

In [27]:
gencode_genes_hg38

,chrom,start,end,name,score,strand,ens_gene
0,chr1,11121,24894,ENSG00000290825-gene,.,+,ENSG00000290825
6,chr1,12010,13670,ENSG00000223972-gene,.,+,ENSG00000223972
14,chr1,14356,30744,ENSG00000310526-gene,.,-,ENSG00000310526
108,chr1,14696,24886,ENSG00000227232-gene,.,-,ENSG00000227232
120,chr1,17369,17436,ENSG00000278267-gene,.,-,ENSG00000278267
...,...,...,...,...,...,...,...
2225559,chrM,14149,14673,ENSG00000198695-gene,.,-,ENSG00000198695
2225564,chrM,14674,14742,ENSG00000210194-gene,.,-,ENSG00000210194
2225567,chrM,14747,15887,ENSG00000198727-gene,.,+,ENSG00000198727
2225572,chrM,15888,15953,ENSG00000210195-gene,.,+,ENSG00000210195


In [34]:
len(set(gencode_genes_hg38["ens_gene"].unique()) - set(fibro_expr["geneID"].unique()))

61792

In [45]:
delta_gencode_ips = set(fibro_expr["geneID"].unique()) - set(gencode_genes_hg38["ens_gene"].unique())
len(delta_gencode_ips)

4109

In [52]:
len(fibro_expr["geneID"].unique())

21041

In [58]:
4109 / 21041 * 100

19.52853951808374

In [51]:
delta_gencode_ips

{'ENSG00000227558',
 'CATG00000040419',
 'CATG00000088944',
 'CATG00000059733',
 'CATG00000016106',
 'CATG00000115481',
 'CATG00000105394',
 'CATG00000013627',
 'CATG00000118331',
 'CATG00000039975',
 'CATG00000077935',
 'CATG00000051831',
 'CATG00000049624',
 'CATG00000021485',
 'CATG00000038597',
 'CATG00000053835',
 'CATG00000094965',
 'CATG00000008376',
 'CATG00000004324',
 'CATG00000061378',
 'ENSG00000187145',
 'CATG00000117605',
 'CATG00000096116',
 'ENSG00000167383',
 'CATG00000009172',
 'CATG00000114610',
 'CATG00000032105',
 'CATG00000034711',
 'CATG00000065968',
 'CATG00000101938',
 'CATG00000026292',
 'ENSG00000261567',
 'CATG00000057795',
 'ENSG00000237975',
 'CATG00000107758',
 'CATG00000078061',
 'CATG00000044236',
 'ENSG00000262888',
 'CATG00000016987',
 'CATG00000106151',
 'CATG00000057831',
 'ENSG00000244268',
 'CATG00000073694',
 'CATG00000052862',
 'CATG00000052311',
 'CATG00000083467',
 'CATG00000025536',
 'CATG00000045588',
 'CATG00000002643',
 'ENSG00000244952',


In [46]:
len(set(gencode_genes_hg38["ens_gene"].unique()) - set(ips_expr["DEG.geneID"].unique()))

68157

In [54]:
delta_gencode_fibro = set(ips_expr["DEG.geneID"].unique()) - set(gencode_genes_hg38["ens_gene"].unique())
len(delta_gencode_fibro)

301

In [55]:
len(ips_expr["DEG.geneID"].unique())

10868

In [56]:
301 / 10868 * 100

2.7695988222304013

In [7]:
to_l2 = FANTOM / 'fantom_l2_genes-hg38-sort.bed'
fantom_l2 = pd.read_csv(to_l2, sep="\t",  names=["chrom","start","end", "name", "score", "strand"], header=0)

In [8]:
fantom_l2

,chrom,start,end,name,score,strand
0,chr1,273882,274416,CATG00000042731-gene,.,-
1,chr1,588120,634563,CATG00000042732-gene,.,-
2,chr1,627377,629095,ENSG00000223659-gene,.,-
3,chr1,629083,634376,ENSG00000248527-gene,.,+
4,chr1,629209,631743,ENSG00000225972-gene,.,+
...,...,...,...,...,...,...
123991,chrY,56838851,56850959,CATG00000115125-gene,.,-
123992,chrY,56855491,56858320,CATG00000114976-gene,.,+
123993,chrY,56855793,56856102,CATG00000115126-gene,.,-
123994,chrY,56867675,56882339,CATG00000114977-gene,.,+


In [9]:
fantom_l2["ens_name"] = fantom_l2["name"].apply(lambda x: x.split("-")[0])

In [17]:
fantom_l2["ens_name"].nunique()

123996

In [15]:
a = set(ips_expr["DEG.geneID"]) - set(fantom_l2["ens_name"])
len(a)

27

In [19]:
a = set(fibro_expr["geneID"]) - set(fantom_l2["ens_name"])
len(a)

71

In [20]:
ENSG00000034053

{'CATG00000043878',
 'CATG00000072514',
 'CATG00000094421',
 'CATG00000102375',
 'CATG00000105435',
 'CATG00000105449',
 'CATG00000105468',
 'CATGC0000023317',
 'CATGE0000171943',
 'ENSG00000013619',
 'ENSG00000017621',
 'ENSG00000034053',
 'ENSG00000049769',
 'ENSG00000064393',
 'ENSG00000080845',
 'ENSG00000090920',
 'ENSG00000099974',
 'ENSG00000101871',
 'ENSG00000102221',
 'ENSG00000123892',
 'ENSG00000128708',
 'ENSG00000130414',
 'ENSG00000131779',
 'ENSG00000147394',
 'ENSG00000148357',
 'ENSG00000150760',
 'ENSG00000158623',
 'ENSG00000159403',
 'ENSG00000161133',
 'ENSG00000163386',
 'ENSG00000168280',
 'ENSG00000170379',
 'ENSG00000170727',
 'ENSG00000172927',
 'ENSG00000172985',
 'ENSG00000173137',
 'ENSG00000175727',
 'ENSG00000177706',
 'ENSG00000178104',
 'ENSG00000178605',
 'ENSG00000179832',
 'ENSG00000182378',
 'ENSG00000182551',
 'ENSG00000183688',
 'ENSG00000184203',
 'ENSG00000186275',
 'ENSG00000198420',
 'ENSG00000203697',
 'ENSG00000205277',
 'ENSG00000207005',
